In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. LOAD DATA
df = pd.read_csv('Sales-Data.csv')

# 2. STRIP SPACES FROM COLUMN NAMES (Crucial!)
df.columns = df.columns.str.strip()

# DEBUG: Check exactly what names are now
print("Python sees these columns:", list(df.columns))

# 3. ROBUST COLUMN MAPPING
# We use .get() to avoid the KeyError if the column is missing
cols = {
    'date': 'Date' if 'Date' in df.columns else 'date',
    'product': 'Product' if 'Product' in df.columns else 'product',
    'price': 'Price' if 'Price' in df.columns else 'price',
    'quantity': 'Quantity' if 'Quantity' in df.columns else 'quantity',
    'payment': 'Payment Method' if 'Payment Method' in df.columns else 'payment_method',
    'city': 'City' if 'City' in df.columns else 'city',
    'manager': 'Manager' if 'Manager' in df.columns else 'manager'
}

# 4. DATA CLEANING & TYPES
# Force numeric conversion (errors='coerce' turns text into NaN)
df[cols['price']] = pd.to_numeric(df[cols['price']], errors='coerce')
df[cols['quantity']] = pd.to_numeric(df[cols['quantity']], errors='coerce')

# Drop rows where Price or Quantity are NaN to prevent math errors
df.dropna(subset=[cols['price'], cols['quantity']], inplace=True)

# Calculate Revenue
df['Revenue'] = df[cols['price']] * df[cols['quantity']]

# Date conversion
df[cols['date']] = pd.to_datetime(df[cols['date']], errors='coerce')
df['Month'] = df[cols['date']].dt.month


# Q.1 Preferred Payment
print("\nQ.1 Most Preferred Payment Method:", df[cols['payment']].value_counts().idxmax())

# Q.2 Most Selling
top_qty_prod = df.groupby(cols['product'])[cols['quantity']].sum().idxmax()
top_rev_prod = df.groupby(cols['product'])['Revenue'].sum().idxmax()
print(f"Q.2 Most Sold by Qty: {top_qty_prod} | By Revenue: {top_rev_prod}")

# Q.3 Max Revenue
print(f"Q.3 Top City: {df.groupby(cols['city'])['Revenue'].sum().idxmax()}")
print(f"    Top Manager: {df.groupby(cols['manager'])['Revenue'].sum().idxmax()}")

# Q.4 Date-wise Revenue Chart
plt.figure(figsize=(10, 5))
df.groupby(cols['date'])['Revenue'].sum().plot(kind='line', marker='o', color='orange')
plt.title('Daily Revenue Analysis')
plt.ylabel('Revenue')
plt.show()

# Q.5 & Q.6 Averages
print(f"Q.5 Overall Average Revenue: {df['Revenue'].mean():.2f}")
nov_dec = df[df['Month'].isin([11, 12])]['Revenue'].mean()
print(f"Q.6 Avg Revenue (Nov & Dec): {nov_dec:.2f}")

# Q.7 & Q.8 Std & Var
print("\n--- Q.7 & Q.8 Statistics ---")
print(df[['Revenue', cols['quantity']]].agg(['std', 'var']))

# Q.10 Product Stats
print("\n--- Q.10 Average Stats per Product ---")
product_stats = df.groupby(cols['product']).agg({cols['quantity']: 'mean', 'Revenue': 'mean'})
print(product_stats)